In [ ]:
# Imports

from pathlib import Path
from scipy.io import savemat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re

In [ ]:
# Define participant and set paths

participant = "vp08"

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError: 
    BASE_DIR = Path.cwd().parent
    
INPUT_PATH = BASE_DIR / "analysis" / "participants" / participant / "logs"
OUTPUT_PATH = BASE_DIR / "analysis" / "participants" / participant

os.chdir(INPUT_PATH)

print(BASE_DIR)
print(INPUT_PATH)
print(OUTPUT_PATH)

In [ ]:
# Get logs in dir

logs = [log for log in os.listdir() if log.endswith(".csv")]

print(logs)

In [ ]:
def build_arrays(run_type):
    match run_type:
        case "func":
            condition_order = ["coherent", "incoherent_real", "incoherent_mock", "neutral", "target"]
        case "loc":
            condition_order = ["Face", "scene", "scrambled"]

    columns = len(condition_order)

    onset_array = np.empty((columns,), dtype=np.object_)
    duration_array = np.empty((columns,), dtype=np.object_)
    names_array = np.empty((columns,), dtype=np.object_)

    return condition_order, onset_array, duration_array, names_array

In [ ]:
def write_conditions_and_onsets(log, condition_order, onset_array):
    for i, condition in enumerate(condition_order):
        cond_onst = list(log[log['condition'] == condition]['s2_onset_cor']) if "coherent" in condition_order else list(log[log['condition'] == condition]['time when it onsets'])
        cond_onset_array = np.array([[num] for num in cond_onst])
        cond_onst = [float(str(num).replace(',', '.')) for num in cond_onst]
    
        cond_onset_array = np.array([[num] for num in cond_onst], dtype=float)
        cond_onset_array = cond_onset_array + 4
        onset_array[i] = cond_onset_array

    return onset_array

In [ ]:
def write_durations(duration_array, condition_order):
    duration_list = [0.20 for c in condition_order]

    for i, duration in enumerate(duration_list):
        duration_array[i] = duration

    return duration_array

In [ ]:
def write_names(condition_order, names_array):
    for i, name in enumerate(condition_order):
        names_array[i] = name

    return names_array

In [ ]:
def to_dict(onset_array, durration_array, names_array):
    mat_dict = {
        "onsets": onset_array,
        "names": names_array,
        "durations": duration_array,
    }

    return mat_dict

In [ ]:
def prep_dir(log_id: str):
    match = re.search(r"\d+", log_id)
    
    if match:
        log_number = match.group()
        func_dir = OUTPUT_PATH / f"0{log_number}_func"
    else:
        func_dir = OUTPUT_PATH / "00_loc"
    
    func_dir.mkdir(parents=True, exist_ok=True)

    return func_dir

In [ ]:
def save_mcf(output_dir: Path, mat_dict: dict):
    out_dir = output_dir / "mcf"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "MCF.mat"
    savemat(out_file, mat_dict)

In [ ]:
# Run conversion for all files

for log in logs:
    log_path = INPUT_PATH / log
    df = pd.read_csv(log_path)

    if re.match(r".*localizer.*\.csv", log):
        print("Run localizer conv")
        run_type = "loc"
    else: 
        print("Run func conv")
        run_type = "func"

    condition_order, empty_onset_array, empty_duration_array, empty_name_array = build_arrays(run_type)
    
    onset_array = write_conditions_and_onsets(df, condition_order, empty_onset_array)
    duration_array = write_durations(empty_duration_array, condition_order)
    names_array = write_names(condition_order, empty_name_array)
    
    mat_dict = to_dict(onset_array, duration_array, names_array)
    
    mcf_dir = prep_dir(log)
    save_mcf(mcf_dir, mat_dict)
    
    print(f"MCF for {log} saved to {mcf_dir}.")
    for array, values in mat_dict.items():
        print(f"\narray")
        print("================")
        for value in values[0:5]:
            print(value)

print(f"MCF conversion completed for {logs}.")